# QuantBook Research: Composite EMA-Cross + TrendStocks

## Objectif

Analyser la complémentarité entre les deux AlphaModels avant de créer le composite:

1. **Corrélation des signaux** : Pourcentage de positions UP chaque semaine
2. **Complémentarité temporelle** : Performance pendant les drawdowns
3. **Overlap des signaux** : Les 5 tech stocks sont communs aux deux univers
4. **Proposition d'allocation** : Ratio optimal entre les deux stratégies

---

In [ ]:
# region imports
from AlgorithmImports import *
import pandas as pd
import numpy as np
from scipy import stats
# endregion

## 1. Définition des univers

- **EMA-Cross** : 5 tech stocks (AAPL, MSFT, GOOGL, AMZN, NVDA)
- **TrendStocks** : 15 stocks diversifiés incluant les 5 tech stocks

In [ ]:
# Univers EMA-Cross
ema_tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA"]

# Univers TrendStocks (inclut les 5 tech)
trend_tickers = [
    "AAPL", "MSFT", "GOOGL", "AMZN", "NVDA",  # Tech (communs)
    "JPM", "V", "MA",                            # Financials
    "UNH", "JNJ",                                 # Healthcare
    "XOM", "CVX",                                 # Energy
    "HD", "PG", "KO"                             # Consumer staples
]

print(f"EMA-Cross: {len(ema_tickers)} stocks")
print(f"TrendStocks: {len(trend_tickers)} stocks")
print(f"Overlap: {set(ema_tickers) & set(trend_tickers)}")

## 2. Initialisation du QuantBook et récupération des données (2015-2025)

In [ ]:
qb = QuantBook()
start_date = datetime(2015, 1, 1)
end_date = datetime(2025, 1, 1)

# Ajouter tous les tickers
for ticker in trend_tickers:
    qb.add_equity(ticker, Resolution.DAILY)

print(f"QuantBook initialisé avec {len(qb.securities)} securities")

## 3. Calcul des indicateurs EMA-Cross

In [ ]:
# Paramètres EMA-Cross
ema_fast = 20
ema_slow = 50

# Récupérer l'historique
history = qb.history(qb.securities.keys(), start_date, end_date, Resolution.DAILY)

# Calculer les signaux EMA-Cross pour les 5 tech stocks
ema_signals = pd.DataFrame(index=history.index.get_level_values(0).unique())

for ticker in ema_tickers:
    symbol = qb.symbol(ticker)
    df_ticker = history.loc[symbol]['close']
    
    # Calculer EMAs
    ema_f = df_ticker.ewm(span=ema_fast, adjust=False).mean()
    ema_s = df_ticker.ewm(span=ema_slow, adjust=False).mean()
    
    # Signal: UP si fast > slow
    ema_signals[ticker] = (ema_f > ema_s).astype(int)

# Pourcentage de stocks UP par semaine
ema_signals_weekly = ema_signals.resample('W').mean()
ema_up_pct = ema_signals_weekly.mean(axis=1) * 100

print(f"Signal EMA-Cross - Moyenne hebdomadaire: {ema_up_pct.mean():.1f}% des stocks UP")
print(f"Ecart-type: {ema_up_pct.std():.1f}%")

## 4. Calcul des indicateurs TrendStocks

In [ ]:
# Paramètres TrendStocks
trend_ema_fast = 20
trend_ema_slow = 50
trend_sma_trend = 200

# Calculer les signaux TrendStocks pour les 15 stocks
trend_signals = pd.DataFrame(index=history.index.get_level_values(0).unique())

for ticker in trend_tickers:
    symbol = qb.symbol(ticker)
    df_ticker = history.loc[symbol]['close']
    
    # Calculer indicateurs
    ema_f = df_ticker.ewm(span=trend_ema_fast, adjust=False).mean()
    ema_s = df_ticker.ewm(span=trend_ema_slow, adjust=False).mean()
    sma_t = df_ticker.rolling(window=trend_sma_trend).mean()
    
    # Signal: UP si Price > SMA200 ET EMA20 > EMA50
    price_above_sma = df_ticker > sma_t
    ema_bullish = ema_f > ema_s
    
    trend_signals[ticker] = (price_above_sma & ema_bullish).astype(int)

# Pourcentage de stocks UP par semaine
trend_signals_weekly = trend_signals.resample('W').mean()
trend_up_pct = trend_signals_weekly.mean(axis=1) * 100

print(f"Signal TrendStocks - Moyenne hebdomadaire: {trend_up_pct.mean():.1f}% des stocks UP")
print(f"Ecart-type: {trend_up_pct.std():.1f}%")

## 5. Analyse 1: Corrélation des signaux (Pearson)

In [ ]:
# Aligner les dates
common_dates = ema_up_pct.index.intersection(trend_up_pct.index)
ema_aligned = ema_up_pct.loc[common_dates]
trend_aligned = trend_up_pct.loc[common_dates]

# Corrélation de Pearson
correlation, p_value = stats.pearsonr(ema_aligned, trend_aligned)

print(f"=== CORRÉLATION DES SIGNAUX (2015-2025) ===")
print(f"Corrélation de Pearson: {correlation:.3f}")
print(f"P-value: {p_value:.2e}")
print(f"")
print(f"Interprétation:")
if correlation > 0.7:
    print(f"  - Forte corrélation positive: les stratégies réagissent souvent de manière similaire")
elif correlation > 0.3:
    print(f"  - Corrélation modérée: certaine complémentarité possible")
elif correlation > 0:
    print(f"  - Faible corrélation: bonne diversification")
else:
    print(f"  - Corrélation négative ou nulle: excellente diversification")

## 6. Analyse 2: Complémentarité temporelle (Drawdowns)

In [ ]:
# Périodes de drawdown majeures du marché (SPY)
drawdown_periods = [
    ("2018 Q4", datetime(2018, 10, 1), datetime(2019, 1, 1)),
    ("2020 Mars", datetime(2020, 2, 20), datetime(2020, 4, 1)),
    ("2022 Bear", datetime(2022, 1, 1), datetime(2022, 12, 31)),
]

print(f"=== PERFORMANCE PENDANT LES DRAWDOWS ===")
print(f"")

for name, start, end in drawdown_periods:
    # Filtrer les données de la période
    mask = (ema_aligned.index >= start) & (ema_aligned.index <= end)
    
    if mask.sum() > 0:
        ema_dd = ema_aligned[mask].mean()
        trend_dd = trend_aligned[mask].mean()
        
        print(f"{name}:")
        print(f"  - EMA-Cross: {ema_dd:.1f}% des stocks UP")
        print(f"  - TrendStocks: {trend_dd:.1f}% des stocks UP")
        print(f"  - Différence: {abs(ema_dd - trend_dd):.1f} points de pourcentage")
        print(f"")

## 7. Analyse 3: Overlap des signaux (5 tech stocks communs)

In [ ]:
# Comparer les signaux pour les 5 tech stocks communs
common_tickers = set(ema_tickers) & set(trend_tickers)

overlap_results = {}

for ticker in common_tickers:
    # Signaux EMA-Cross pour ce ticker
    ema_sig = ema_signals[ticker].resample('W').last()
    
    # Signaux TrendStocks pour ce ticker
    trend_sig = trend_signals[ticker].resample('W').last()
    
    # Aligner
    common = ema_sig.index.intersection(trend_sig.index)
    ema_common = ema_sig.loc[common]
    trend_common = trend_sig.loc[common]
    
    # Scénarios:
    # - UP/UP: Les deux sont haussiers
    # - UP/FLAT ou FLAT/UP: Divergence
    # - FLAT/FLAT: Les deux sont neutres
    
    up_up = ((ema_common == 1) & (trend_common == 1)).sum()
    up_flat = ((ema_common == 1) & (trend_common == 0)).sum()
    flat_up = ((ema_common == 0) & (trend_common == 1)).sum()
    flat_flat = ((ema_common == 0) & (trend_common == 0)).sum()
    total = len(common)
    
    overlap_results[ticker] = {
        'up_up_pct': up_up / total * 100,
        'divergence_pct': (up_flat + flat_up) / total * 100,
        'flat_flat_pct': flat_flat / total * 100,
    }

print(f"=== OVERLAP DES SIGNAUX (5 TECH STOCKS) ===")
print(f"")
print(f"{'Ticker':<8} {'UP/UP':>10} {'Divergence':>12} {'FLAT/FLAT':>12}")
print(f"-" * 46)

for ticker, stats in overlap_results.items():
    print(f"{ticker:<8} {stats['up_up_pct']:>9.1f}% {stats['divergence_pct']:>11.1f}% {stats['flat_flat_pct']:>11.1f}%")

# Moyennes
avg_up_up = np.mean([s['up_up_pct'] for s in overlap_results.values()])
avg_div = np.mean([s['divergence_pct'] for s in overlap_results.values()])
avg_flat = np.mean([s['flat_flat_pct'] for s in overlap_results.values()])
print(f"-" * 46)
print(f"{'MOYENNE':<8} {avg_up_up:>9.1f}% {avg_div:>11.1f}% {avg_flat:>11.1f}%")

## 8. Analyse 4: Proposition d'allocation

In [ ]:
print(f"=== PROPOSITION D'ALLOCATION ===")
print(f"")
print(f"Synthèse des analyses:")
print(f"1. Corrélation: {correlation:.3f}")
print(f"2. Volatilité EMA-Cross: {ema_up_pct.std():.1f}%")
print(f"3. Volatilité TrendStocks: {trend_up_pct.std():.1f}%")
print(f"4. Divergence moyenne sur tech stocks: {avg_div:.1f}%")
print(f"")
print(f"Recommandation:")

# Décision basée sur la corrélation
if correlation > 0.5:
    # Forte corrélation: favoriser la plus robuste (TrendStocks avec confirmation double)
    print(f"  - EMA30/Trend70: Corrélation élevée, TrendStocks plus robuste")
    recommended = "EMA30/Trend70"
elif correlation > 0.2:
    # Corrélation modérée: équilibrer
    print(f"  - EMA40/Trend60: Corrélation modérée, bon équilibre")
    recommended = "EMA40/Trend60"
else:
    # Faible corrélation: diversifier équitablement
    print(f"  - EMA50/Trend50: Faible corrélation, diversification optimale")
    recommended = "EMA50/Trend50"

print(f"")
print(f"Allocation de départ recommandée: {recommended}")
print(f"")
print(f"Plan de sweep (à exécuter après backtest initial):")
print(f"  - EMA30/Trend70")
print(f"  - EMA40/Trend60")
print(f"  - EMA50/Trend50")
print(f"  - EMA60/Trend40")
print(f"  - EMA70/Trend30")

## 9. Conclusion

Ce QuantBook a permis d'analyser la complémentarité entre EMA-Cross et TrendStocks.
Les résultats guideront l'allocation initiale du composite ainsi que les sweeps à effectuer.